# ML6402 – Practical 03: Data Pre-processing

**Instructor:** Dr. Ahmed Sabbah  
**Course:** Machine Learning (ML6402)

This practical turns the concepts from Lecture 3 into an executable machine-learning workflow.

### Learning outcomes

By the end of this practical, you should be able to:

1. inspect a raw dataset and identify common quality problems;
2. separate features and target variables;
3. split data into training and test sets correctly;
4. handle missing numerical values;
5. encode categorical attributes;
6. apply normalization and standardization appropriately;
7. explain and prevent data leakage;
8. build a reproducible preprocessing workflow using `ColumnTransformer` and `Pipeline`.

> **Important idea:** preprocessing is not just cleaning. It is part of the experimental design.

## Part A – Guided preprocessing example

We begin with a deliberately small dataset so every transformation can be inspected manually.

The dataset contains four attributes:

- **City** – categorical input feature
- **Age** – numerical input feature
- **Salary** – numerical input feature
- **Purchased** – target variable

Some values are missing on purpose so that we can practice imputation.

### Choose how to load the dataset

This notebook supports **two execution environments**:

1. **Local Anaconda / Jupyter** – load `salary.csv` from the local `data` folder.
2. **Google Colab** – load the same dataset directly from this GitHub repository.

**Use only one method at a time.** In the next code cell, uncomment the method you want to use and keep the other method commented.

> If you cloned or downloaded the complete repository and opened the notebook locally, use **Method 1**. If you opened only the notebook in Google Colab, use **Method 2**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# DATASET LOADING – CHOOSE ONE METHOD ONLY
# ============================================================

# METHOD 1: Local Anaconda / Jupyter
# Use this when you cloned/downloaded the complete repository.
DATA_PATH = Path("data") / "salary.csv"
df = pd.read_csv(DATA_PATH)

# METHOD 2: Google Colab
# If you are using Colab, COMMENT Method 1 above and UNCOMMENT
# the two lines below.
# DATA_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture03_Data_Preprocessing/data/salary.csv"
# df = pd.read_csv(DATA_URL)

# Display the dataset
df

### 1. Inspect the data before changing anything

Before preprocessing, we should understand:

- the number of rows and columns;
- column names;
- data types;
- missing values;
- duplicated rows;
- the distribution of important numeric features.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicated rows:", df.duplicated().sum())

In [ ]:
df.describe(include="all")

### Question 1

Look at the output above.

- Which two features contain missing values?
- How many values are missing in each?
- Would dropping the affected rows be a good idea for a dataset this small? Why or why not?

## 2. Separate input features and target

The machine-learning model will use:

- `X` for input features;
- `y` for the target variable.

Keeping feature names in a Pandas DataFrame makes the preprocessing steps easier to understand than immediately converting everything to a NumPy array.

In [ ]:
X = df.drop(columns="Purchased")
y = df["Purchased"]

print("X:")
display(X)

print("\ny:")
display(y)

## 3. Split before learning preprocessing parameters

A common mistake is to calculate means, medians, scaling parameters, or encodings from the **full dataset** before the split.

That allows information from the test set to influence training.

This is **data leakage**.

The correct order is:

1. split into training and test sets;
2. fit preprocessing transformations using the training set only;
3. apply the fitted transformations to both training and test data.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
print("Training data:")
display(X_train)

print("\nTest data:")
display(X_test)

### Question 2

Why is `random_state` useful in a teaching or research experiment?

Why should the test set remain untouched while preprocessing decisions are learned?

## 4. Numerical imputation

The numerical columns are:

- `Age`
- `Salary`

We will use **median imputation**.

Why median?

- it is simple;
- it is less sensitive to extreme values than the mean;
- it is a reasonable default when a numeric distribution may be skewed.

The imputer is **fitted on the training data only**.

In [ ]:
from sklearn.impute import SimpleImputer

numeric_features = ["Age", "Salary"]

num_imputer = SimpleImputer(strategy="median")

X_train_num = num_imputer.fit_transform(X_train[numeric_features])
X_test_num = num_imputer.transform(X_test[numeric_features])

print("Learned medians:", num_imputer.statistics_)
print("\nTransformed training numeric data:")
print(X_train_num)

### Verify the learned values

The values in `num_imputer.statistics_` were learned only from the training subset.

That is exactly what we want in a valid experiment.

## 5. Categorical encoding

`City` is a **nominal** variable. The city names do not have a natural numeric order.

Therefore, directly mapping cities to 0, 1, 2 could accidentally suggest an ordering that does not exist.

We use **one-hot encoding** instead.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_city = cat_encoder.fit_transform(X_train[["City"]])
X_test_city = cat_encoder.transform(X_test[["City"]])

print("Categories learned from training data:")
print(cat_encoder.categories_)

print("\nEncoded training values:")
print(X_train_city)

### Why `handle_unknown="ignore"`?

A future sample may contain a category that did not appear in the training set.

This option prevents the transform step from failing when that happens.

## 6. Standardization and normalization

Two common scaling approaches are:

### Standardization

[
z = \frac{x - \mu}{\sigma}
]

The transformed feature is centered around 0 and scaled by its standard deviation.

### Min–Max normalization

[
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
]

The values are mapped into a fixed range, usually 0 to 1.

We will compare both approaches on the training data.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# First impute the numeric training data
train_num_imputed = pd.DataFrame(
    num_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index
)

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

train_standardized = standard_scaler.fit_transform(train_num_imputed)
train_normalized = minmax_scaler.fit_transform(train_num_imputed)

print("Original / imputed numeric data:")
display(train_num_imputed)

print("Standardized:")
display(pd.DataFrame(train_standardized, columns=numeric_features, index=X_train.index))

print("Normalized:")
display(pd.DataFrame(train_normalized, columns=numeric_features, index=X_train.index))

### Question 3

Compare the standardized and normalized values.

- Which method forces values into the range 0–1?
- Which method produces values centered around 0?
- Why can Min–Max normalization be strongly affected by outliers?

## 7. Build a single preprocessing object

In a real machine-learning project, manually transforming each column separately becomes difficult to reproduce.

`ColumnTransformer` allows us to define different transformations for different groups of columns.

We will create:

- a numeric pipeline: **median imputation → standardization**
- a categorical pipeline: **most-frequent imputation → one-hot encoding**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ["Age", "Salary"]
categorical_features = ["City"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

preprocessor

## 8. Fit on training data, transform both sets

This is the central workflow for this practical.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print(feature_names)

### Key interpretation

Notice that preprocessing may change the number of columns.

Originally we had three input features:

- City
- Age
- Salary

After one-hot encoding, City becomes multiple binary columns.

This is why preprocessing is not merely cosmetic: it changes the representation the model actually receives.

## 9. Add a simple model to complete the workflow

The goal here is **not** to study Logistic Regression in depth yet.

We only use a simple classifier to show how preprocessing and a model can be combined into one reproducible pipeline.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression())
])

model.fit(X_train, y_train)

print("Training accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

### Important caution

The dataset contains only 10 observations.

Therefore, the accuracy above should **not** be interpreted as evidence that the model is good.

This example is for understanding the mechanics of preprocessing and reproducibility.

Later practicals will use larger datasets for meaningful model evaluation.

## 10. What would leakage look like?

The following pattern is conceptually wrong:

```python
# WRONG IDEA
imputer.fit(full_dataset)
scaler.fit(full_dataset)
train_test_split(...)
```

Why?

Because the preprocessing steps have already learned information from rows that later belong to the test set.

A valid experiment learns all data-dependent preprocessing parameters from the training set only.

## Part B – Student exercises

Complete the following tasks without looking back at the earlier cells unless needed.

### Exercise 1 – Missing values

1. Reload the dataset into a new variable.
2. Calculate the percentage of missing values in each column.
3. Explain whether you would use mean or median imputation for `Salary`.
4. Justify your answer.

### Exercise 2 – Categorical encoding

1. Print all unique values in `City`.
2. Apply one-hot encoding.
3. Explain why label encoding could be misleading for this feature.

### Exercise 3 – Scaling

1. Apply `MinMaxScaler` to `Age` and `Salary`.
2. Apply `StandardScaler` to the same columns.
3. Compare the outputs.
4. Explain which scaler you would prefer if a strong outlier were present.

### Exercise 4 – Leakage

Write two short code snippets:

- one that introduces leakage;
- one that avoids leakage.

Then explain the difference in your own words.

## Challenge – Build the complete preprocessing pipeline yourself

Create a new preprocessing pipeline from scratch with these requirements:

- numeric columns: `Age`, `Salary`
- categorical column: `City`
- numeric missing values: median
- categorical missing values: most frequent
- numeric scaling: standardization
- categorical encoding: one-hot encoding
- unknown categories must not cause an error

Then:

1. split the data;
2. fit the pipeline on training data;
3. transform the test data;
4. print the final feature names;
5. explain why this workflow is reproducible.

In [ ]:
# Write your solution here


## Reflection questions

Answer briefly:

1. Why is preprocessing considered part of the machine-learning experiment?
2. What is data leakage?
3. Why should preprocessing parameters be learned from training data only?
4. Why is one-hot encoding appropriate for nominal categories?
5. What is the difference between standardization and normalization?
6. Why can a very small dataset be useful for learning a technique but poor for evaluating a model?

## Source note

The small preprocessing dataset and the sequence of core preprocessing operations are aligned with the example used in the course's primary textbook. The notebook extends that example with a leakage-safe train/test workflow, modern Scikit-Learn pipelines, reproducibility practices, interpretation questions, and graduate-level experimental considerations.

---

**ML6402 – Machine Learning**  
**Instructor: Dr. Ahmed Sabbah**